# 🏛️ Bremen Company Intelligence — RAG Chatbot Demo
**WHU · Data-Driven Entrepreneurship · SS 2026**

This notebook walks through the full RAG (Retrieval-Augmented Generation) pipeline step by step:

```
User question
     │
     ▼
  [ROUTER]  — classifies intent: lookup / filter / general
     │
     ▼
  [FAISS]   — retrieves the most relevant company documents
     │
     ▼
  [GROQ LLM] — generates a grounded answer from retrieved context
     │
     ▼
  Answer + cited companies
```

**Data:** Agent-collected columns only — no Orbis data  
**LLM:** Groq `llama-3.3-70b-versatile`  
**Embeddings:** `sentence-transformers/all-MiniLM-L6-v2` + FAISS

## Step 1 — Install dependencies

In [ ]:
!pip install -q groq langchain-core langchain-community sentence-transformers faiss-cpu pandas

## Step 2 — Imports

In [ ]:
import os
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from groq import Groq
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('✅ All imports successful')

## Step 3 — Load your Groq API key

In [ ]:
# Reads from groq.txt if present, otherwise paste your key below
GROQ_API_KEY = ""

if os.path.exists('groq.txt'):
    with open('groq.txt') as f:
        GROQ_API_KEY = f.readline().strip()
    print('✅ Key loaded from groq.txt')
elif GROQ_API_KEY:
    print('✅ Key set manually')
else:
    print('⚠️  No key found — paste your Groq key into GROQ_API_KEY above')

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

## Step 4 — Load the agent-collected dataset

We use **only the columns your agent pipeline collected** — no Orbis employee/scaler data.

In [ ]:
DATA_PATH = 'df_agent_progress final(Sheet1).csv'   # ← update if needed

raw = pd.read_csv(DATA_PATH, encoding='latin-1', sep=';', engine='python', on_bad_lines='skip')
print(f'Raw shape: {raw.shape}')
print('Columns:', list(raw.columns))

In [ ]:
# Keep and rename agent-collected columns only
col_map = {
    'company_name':                     'Company',
    'Legal_Form':                        'Legal Form',
    'Industry':                          'Industry',
    'B2B_or_B2C':                        'Customer Model',
    'Company_Description':               'Description',
    'Key_Activities_Product_Offerings':  'Key Activities',
    'nace_section':                      'NACE Section',
    'incorporation_date':                'Founded',
    'city':                              'City',
}

df = raw.rename(columns={k: v for k, v in col_map.items() if k in raw.columns})
df = df[[v for v in col_map.values() if v in df.columns]].copy()
df.replace(['Not found', 'n.a.', 'N/A', '', 'nan'], pd.NA, inplace=True)
df = df[df['Company'].notna()].drop_duplicates(subset=['Company']).reset_index(drop=True)

# Normalise B2B/B2C labels
def clean_b2b(v):
    if pd.isna(v): return 'Unknown'
    v = str(v).strip().lower()
    if v.startswith('b2b') or v == 'business': return 'B2B'
    if v.startswith('b2c'): return 'B2C'
    if 'both' in v: return 'Both'
    return 'Unknown'

df['Customer Model'] = df['Customer Model'].apply(clean_b2b)

print(f'\nClean dataset: {df.shape[0]} companies, {df.shape[1]} columns')
df.head()

In [ ]:
# Quick overview
print('=== Customer Model breakdown ===')
print(df['Customer Model'].value_counts())
print()
print('=== Top 10 Industries ===')
print(df['Industry'].value_counts().head(10))

## Step 5 — Build documents for RAG

Each company becomes one text document. The embeddings model turns these into vectors stored in FAISS.

In [ ]:
def make_document(row) -> Document:
    """Convert one company row into a LangChain Document."""
    lines = [
        f"Company: {row.get('Company', '')}",
        f"Industry: {row.get('Industry', '')}",
        f"Legal Form: {row.get('Legal Form', '')}",
        f"Customer Model: {row.get('Customer Model', '')}",
        f"NACE Section: {row.get('NACE Section', '')}",
        f"Founded: {row.get('Founded', '')}",
        f"Description: {row.get('Description', '')}",
        f"Key Activities: {row.get('Key Activities', '')}",
    ]
    text = '\n'.join(l for l in lines if not l.endswith(': ') and 'nan' not in l.lower())
    return Document(page_content=text, metadata={'company': row.get('Company', '')})

docs = [make_document(row) for _, row in df.iterrows()]
print(f'Built {len(docs)} documents')
print()
print('=== Example document ===')
print(docs[0].page_content)

## Step 6 — Build the FAISS vector index

`all-MiniLM-L6-v2` converts each document into a 384-dimensional vector.  
FAISS stores them so we can find the closest matches to any query in milliseconds.

In [ ]:
print('Loading embedding model…')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

print('Building FAISS index…')
vectorstore = FAISS.from_documents(docs, embeddings)
print(f'✅ Index built — {len(docs)} vectors stored')

## Step 7 — Test retrieval

Given any question, FAISS returns the most semantically similar company documents.

In [ ]:
def retrieve(question: str, k: int = 4):
    """Return top-k most relevant documents for a question."""
    results = vectorstore.similarity_search(question, k=k)
    print(f'Query: "{question}"')
    print(f'Top {k} results:\n')
    for i, doc in enumerate(results, 1):
        print(f'[{i}] {doc.metadata["company"]}')
        print(doc.page_content[:200])
        print()
    return results

results = retrieve('shipping and logistics company')

In [ ]:
# Try another query
results = retrieve('software and technology B2B')

## Step 8 — The Router

The Router classifies each question into one of three intents:
- **lookup** — asking about a specific company
- **filter** — listing/comparing companies by a criterion
- **general** — open question about the dataset

In [ ]:
def route(question: str) -> str:
    """Classify question intent using Groq (falls back to regex)."""
    if client:
        resp = client.chat.completions.create(
            model='llama-3.3-70b-versatile',
            messages=[
                {'role': 'system', 'content':
                    "Classify the question. Reply with ONLY one word: "
                    "'lookup' (specific company), "
                    "'filter' (list/compare companies by a criterion), "
                    "or 'general' (anything else)."},
                {'role': 'user', 'content': f'Question: {question}'}
            ],
            temperature=0,
            max_tokens=5,
        )
        intent = resp.choices[0].message.content.strip().lower().strip("'")
        if intent in ('lookup', 'filter', 'general'):
            return intent

    # Regex fallback (no API key needed)
    q = question.lower()
    if re.search(r'(tell me about|who is|what does|describe|is .+ a)', q): return 'lookup'
    if re.search(r'(list|show|find|which companies|all companies|filter|b2b|b2c)', q): return 'filter'
    return 'general'

# Test the router
test_questions = [
    'Tell me about ArcelorMittal Bremen',
    'List all B2B logistics companies',
    'What industries are most common in the dataset?',
    'Which companies are in real estate?',
    'Describe Roehlig Logistics',
]

print(f'{"Question":<50} {"Intent"}')
print('-' * 60)
for q in test_questions:
    print(f'{q:<50} {route(q)}')

## Step 9 — The Agents

Each agent handles one intent type. It retrieves relevant docs, builds a prompt, and calls the LLM.

In [ ]:
def ask_llm(system: str, user: str) -> str:
    """Call Groq LLM with a system + user message."""
    if not client:
        return '(No API key — add your Groq key in Step 3 for LLM answers)'
    resp = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
        temperature=0.3,
        max_tokens=500,
    )
    return resp.choices[0].message.content.strip()

print('✅ LLM helper ready')

In [ ]:
# ── LOOKUP AGENT ──────────────────────────────────────────────────────────────
def lookup_agent(question: str):
    """Look up a specific company."""
    docs = vectorstore.similarity_search(question, k=4)
    cited = [d.metadata['company'] for d in docs]
    context = '\n\n---\n'.join(d.page_content for d in docs)

    answer = ask_llm(
        system=(
            'You are a Bremen company analyst. Use ONLY the context below. '
            'Answer in 3–5 sentences. Name the company, its industry, legal form, '
            'customer model, and what it does.'
        ),
        user=f'Context:\n{context}\n\nQuestion: {question}'
    )
    return answer, cited


# ── FILTER AGENT ──────────────────────────────────────────────────────────────
def filter_agent(question: str):
    """Filter and list companies by a criterion."""
    q = question.lower()
    filtered = df.copy()
    filter_desc = []

    if 'b2b' in q:
        filtered = filtered[filtered['Customer Model'] == 'B2B']; filter_desc.append('B2B')
    elif 'b2c' in q:
        filtered = filtered[filtered['Customer Model'] == 'B2C']; filter_desc.append('B2C')

    kws = re.findall(r'(logistics|tech|software|real estate|construction|finance|health|'
                     r'education|manufacturing|retail|transport|hospitality|shipping|'
                     r'insurance|consulting|engineering|energy|media)', q)
    if kws:
        pattern = '|'.join(kws)
        mask = filtered['Industry'].str.contains(pattern, case=False, na=False)
        if mask.sum() > 0:
            filtered = filtered[mask]; filter_desc.append(f'industry: {", ".join(kws)}')

    for lf in ['gmbh', ' ag ', ' kg ']:
        if lf in q:
            mask = filtered['Legal Form'].str.contains(lf.strip(), case=False, na=False)
            if mask.sum() > 0:
                filtered = filtered[mask]; filter_desc.append(f'legal: {lf.strip().upper()}')

    n = len(filtered)
    sample = filtered.head(8)
    b2b_n = (filtered['Customer Model'] == 'B2B').sum()
    b2c_n = (filtered['Customer Model'] == 'B2C').sum()
    rows_text = '\n'.join(
        f"• {r['Company']} | {r.get('Industry','—')} | {r.get('Legal Form','—')} | {r.get('Customer Model','—')}"
        for _, r in sample.iterrows()
    )

    ctx = (f"Found {n} companies" + (f" matching: {', '.join(filter_desc)}" if filter_desc else "") +
           f". B2B: {b2b_n}, B2C: {b2c_n}.\n\nSample:\n{rows_text}")

    answer = ask_llm(
        system='You are a Bremen company analyst. Summarise the results clearly. Mention count, B2B/B2C split, and any interesting pattern.',
        user=f'{ctx}\n\nUser asked: {question}'
    )
    cited = sample['Company'].tolist()
    return answer, cited


# ── GENERAL AGENT ─────────────────────────────────────────────────────────────
def general_agent(question: str):
    """Answer open questions using retrieved context + dataset stats."""
    docs = vectorstore.similarity_search(question, k=6)
    cited = [d.metadata['company'] for d in docs]
    context = '\n\n---\n'.join(d.page_content for d in docs)

    n = len(df)
    b2b_pct = round((df['Customer Model'] == 'B2B').sum() / n * 100, 1)
    b2c_pct = round((df['Customer Model'] == 'B2C').sum() / n * 100, 1)
    top_ind = df['Industry'].value_counts().head(5).to_dict()
    stats = (f'Dataset: {n} Bremen companies. B2B: {b2b_pct}%, B2C: {b2c_pct}%. '
             f'Top industries: {", ".join(f"{k} ({v})" for k,v in top_ind.items())}.')

    answer = ask_llm(
        system=f'You are a Bremen company analyst for WHU DDE. {stats} Be concise and specific.',
        user=f'Context:\n{context}\n\nQuestion: {question}'
    )
    return answer, cited

print('✅ All three agents ready')

## Step 10 — Full pipeline demo

Now let's wire everything together and ask real questions.

In [ ]:
def chat(question: str):
    """Full RAG pipeline: route → retrieve → answer."""
    print(f'❓ Question: {question}')
    print()

    intent = route(question)
    print(f'⌥ Router → {intent.upper()}')
    print()

    if intent == 'lookup':
        answer, cited = lookup_agent(question)
    elif intent == 'filter':
        answer, cited = filter_agent(question)
    else:
        answer, cited = general_agent(question)

    print(f'💬 Answer:')
    print(answer)
    print()
    print(f'📌 Cited companies: {", ".join(cited[:4])}')
    print('─' * 70)
    return answer

In [ ]:
# Demo 1 — Lookup a specific company
_ = chat('Tell me about ArcelorMittal Bremen')

In [ ]:
# Demo 2 — Filter by industry + customer model
_ = chat('List all B2B logistics companies')

In [ ]:
# Demo 3 — General / open question
_ = chat('What industries are most common in the Bremen dataset?')

In [ ]:
# Demo 4 — Your own question
_ = chat('Which companies are in real estate?')

In [ ]:
# Demo 5 — Try any question here
my_question = 'Find software companies that work with businesses'
_ = chat(my_question)

## Summary

| Component | Technology | Role |
|---|---|---|
| **Data** | Agent-collected CSV (911 companies) | Source of truth — no Orbis columns |
| **Documents** | LangChain `Document` objects | One per company, all fields as text |
| **Embeddings** | `all-MiniLM-L6-v2` (HuggingFace) | Converts text → 384-dim vectors |
| **Vector store** | FAISS | Finds nearest neighbours in milliseconds |
| **Router** | Groq `llama-3.3-70b-versatile` | Classifies intent: lookup / filter / general |
| **LLM** | Groq `llama-3.3-70b-versatile` | Generates grounded answers from retrieved context |

The full Streamlit app is in **`chatbot.py`** — run with `streamlit run chatbot.py`.